# Decision Planner Comparison

Colab notebook for comparing seven decision planning strategies on a small CICIDS sample.

In [ ]:
!pip install numpy scikit-learn transformers -q

In [ ]:
!git clone https://github.com/Lawapaul/AI_Agentic_DL.git

import os
import sys

repo_path = '/content/AI_Agentic_DL'
sys.path.insert(0, repo_path)

In [ ]:
%cd /content/AI_Agentic_DL
!git fetch origin
!git checkout codex/decision-planner-experiments
!git pull origin codex/decision-planner-experiments

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import numpy as np

path = '/content/drive/MyDrive/Deep Learning Project/AI Agentic/data/processed/'

X = np.load(path + 'X_test.npy')[:20]
y = np.load(path + 'y_test.npy')[:20]

X = X.reshape(X.shape[0], -1)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=10, random_state=42)
model.fit(X, y)

preds = model.predict(X)
probs = model.predict_proba(X)

In [ ]:
from decision_planner.rule_based import RuleBasedPlanner
from decision_planner.confidence_based import ConfidenceBasedPlanner
from decision_planner.risk_based import RiskBasedPlanner
from decision_planner.hybrid import HybridPlanner
from decision_planner.rl_planner import RLPlanner
from decision_planner.llm_planner import LLMPlanner
from decision_planner.policy_based import PolicyBasedPlanner
from collections import Counter

In [ ]:
def vote(actions):
    return Counter(actions).most_common(1)[0][0]

def clean_result(r):
    return {k: str(v) for k, v in r.items()}

In [ ]:
attacks = ['Attack' if int(pred) != 0 else 'Normal Traffic' for pred in preds[:10]]
confidences = [float(max(prob)) for prob in probs[:10]]
severities = [
    'High' if conf > 0.8 else 'Medium' if conf > 0.6 else 'Low'
    for conf in confidences
]
targets = [
    'No Action' if attack == 'Normal Traffic' else 'Block' if conf > 0.85 else 'Alert' if conf > 0.6 else 'Monitor'
    for attack, conf in zip(attacks, confidences)
]

In [ ]:
rule = RuleBasedPlanner().fit(attacks, confidences, targets)
conf = ConfidenceBasedPlanner().fit(confidences, targets)
risk = RiskBasedPlanner().fit(confidences, severities, targets)
hybrid = HybridPlanner().fit(attacks, confidences, severities, targets)
rl = RLPlanner().fit(attacks, confidences, targets, epochs=10)
llm = LLMPlanner().fit(attacks, confidences, severities, targets)
policy = PolicyBasedPlanner().fit(attacks, severities, targets)

In [ ]:
results = []

for i in range(10):
    attack = attacks[i]
    conf_score = confidences[i]
    severity = severities[i]

    rule_action = rule.decide(attack, conf_score, severity)
    conf_action = conf.decide(attack, conf_score, severity)
    risk_action = risk.decide(attack, conf_score, severity)
    hybrid_action = hybrid.decide(attack, conf_score, severity)
    rl_action = rl.decide(attack, conf_score, severity)
    llm_action = llm.decide(attack, conf_score, severity)
    policy_action = policy.decide(attack, conf_score, severity)
    final_action = vote([rule_action, rl_action, policy_action])

    results.append({
        'index': i,
        'attack': attack,
        'confidence': conf_score,
        'severity': severity,
        'target_action': targets[i],
        'rule': rule_action,
        'confidence_based': conf_action,
        'risk': risk_action,
        'hybrid': hybrid_action,
        'rl': rl_action,
        'llm': llm_action,
        'policy': policy_action,
        'final_action': final_action
    })

    print(results[-1])

results = [clean_result(r) for r in results]

In [ ]:
import json

repo_path = '/content/your-repo-name'
if os.path.exists('/content/AI_Agentic_DL'):
    repo_path = '/content/AI_Agentic_DL'

save_path = os.path.join(repo_path, 'experiments', 'results')
os.makedirs(save_path, exist_ok=True)

with open(os.path.join(save_path, 'final_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print('Saved successfully')